In [1]:
#%pip install torch torchvision

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import joblib
from torch.utils.data import TensorDataset, DataLoader

# ============================================================
# 1. Laden des Datensatzes
# ============================================================
# Achtung: Semikolon als Trennzeichen!
df = pd.read_csv('winequality-white.csv', sep=';')
print(f"Datensatz: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
print(df.head())

Datensatz: 4898 Zeilen, 12 Spalten
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.0              0.27         0.36            20.7      0.045   
1            6.3              0.30         0.34             1.6      0.049   
2            8.1              0.28         0.40             6.9      0.050   
3            7.2              0.23         0.32             8.5      0.058   
4            7.2              0.23         0.32             8.5      0.058   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 45.0                 170.0   1.0010  3.00       0.45   
1                 14.0                 132.0   0.9940  3.30       0.49   
2                 30.0                  97.0   0.9951  3.26       0.44   
3                 47.0                 186.0   0.9956  3.19       0.40   
4                 47.0                 186.0   0.9956  3.19       0.40   

   alcohol  quality  
0      8.8        6  
1      

In [3]:
# ============================================================
# 2. Features (X) und Target (y) trennen
# ============================================================
X = df.drop('quality', axis=1).values       # 11 Features
#y = df['quality'].values                     # Qualitätsstufen 3-9
y_bin = (df['quality'] >= 6).astype(int)    # Binäre Zielvariable: 0=Schlecht, 1=Gut

# LabelEncoder: Qualitätsstufen 3-9 → fortlaufende Indizes 0-6
# (CrossEntropyLoss erwartet Klassen ab 0)
le = LabelEncoder()
y = le.fit_transform(y_bin)
print(f"Klassen (original):  {le.classes_}")
print(f"Klassen (kodiert):   {list(range(len(le.classes_)))}")
print(f"Anzahl Klassen:      {len(le.classes_)}")

Klassen (original):  [0 1]
Klassen (kodiert):   [0, 1]
Anzahl Klassen:      2


In [4]:
# ============================================================
# 3. Train/Val/Test-Split
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=0, stratify=y_train
)

print(f"Train: {X_train.shape[0]}  |  Val: {X_val.shape[0]}  |  Test: {X_test.shape[0]}")

Train: 3134  |  Val: 784  |  Test: 980


In [5]:
# ============================================================
# 4. Standardisierung (z-Score) der Features
# ============================================================
# WICHTIG: fit() nur auf Trainingsdaten, transform() auf alle Splits
# Grund: Val/Test sollen Bedingungen simulieren, unter denen das
#         Modell auf neue, ungesehene Daten trifft.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # fit + transform
X_val   = scaler.transform(X_val)         # nur transform
X_test  = scaler.transform(X_test)        # nur transform

# Kontrolle: Mittelwert ≈ 0 und Std ≈ 1 auf Trainingsdaten
print("Train-Mittelwerte (soll ≈ 0):")
print(np.round(X_train.mean(axis=0), 4))
print("Train-Std (soll ≈ 1):")
print(np.round(X_train.std(axis=0), 4))

Train-Mittelwerte (soll ≈ 0):
[ 0. -0.  0.  0. -0.  0. -0.  0.  0. -0. -0.]
Train-Std (soll ≈ 1):
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [6]:
# ============================================================
# 5. Umwandeln der Daten in PyTorch-Tensoren
# ============================================================
# from_numpy() teilt Speicher mit NumPy (keine Kopie) – effizienter als torch.tensor()
X_train_t = torch.from_numpy(X_train).float()
X_val_t   = torch.from_numpy(X_val).float()
X_test_t  = torch.from_numpy(X_test).float()
y_train_t = torch.from_numpy(y_train).long()
y_val_t   = torch.from_numpy(y_val).long()
y_test_t  = torch.from_numpy(y_test).long()

In [7]:
# ============================================================
# 6. DataLoader für Batch-Verarbeitung
# ============================================================
train_dataset = TensorDataset(X_train_t, y_train_t)
batch_size = 64
# shuffle=True sorgt dafür, dass die Daten in jeder Epoche gemischt werden
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [8]:
# ============================================================
# 7. Neuronales Netz definieren
# ============================================================
n_features = X_train.shape[1]   
n_classes  = len(le.classes_)

net = nn.Sequential(
    nn.Linear(n_features, 32),
    nn.ReLU(),
    nn.Linear(32, n_classes)
)
print(net)

Sequential(
  (0): Linear(in_features=11, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=2, bias=True)
)


In [ ]:
# ============================================================
# 8. Training mit Batch-Verarbeitung
# ============================================================
criterion = nn.CrossEntropyLoss() # BCELoss()
optimizer = optim.AdamW(net.parameters(), lr=0.005)
#optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)  # SGD mit Momentum

n_epochs = 100

net.train()
for epoch in range(n_epochs):
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = net(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

    # Validierung nach jeder Epoche
    net.eval()
    with torch.no_grad():
        val_out  = net(X_val_t)
        val_loss = criterion(val_out, y_val_t)
        _, val_pred = torch.max(val_out, 1)
        val_acc = accuracy_score(y_val_t, val_pred)
    net.train()

    if epoch % 10 == 0 or epoch == n_epochs - 1:
        print(f'Epoch {epoch:3d} | Train-Loss: {loss.item():.4f} | Val-Loss: {val_loss.item():.4f} | Val-Acc: {val_acc:.4f}')

Epoch   0 | Train-Loss: 0.4309 | Val-Loss: 0.4902 | Val-Acc: 0.7717
Epoch  10 | Train-Loss: 0.4763 | Val-Loss: 0.4727 | Val-Acc: 0.7730
Epoch  20 | Train-Loss: 0.4391 | Val-Loss: 0.4901 | Val-Acc: 0.7589
Epoch  30 | Train-Loss: 0.4070 | Val-Loss: 0.4932 | Val-Acc: 0.7679
Epoch  40 | Train-Loss: 0.5007 | Val-Loss: 0.4975 | Val-Acc: 0.7602
Epoch  50 | Train-Loss: 0.3263 | Val-Loss: 0.4918 | Val-Acc: 0.7653
Epoch  60 | Train-Loss: 0.4321 | Val-Loss: 0.5086 | Val-Acc: 0.7602
Epoch  70 | Train-Loss: 0.4429 | Val-Loss: 0.4938 | Val-Acc: 0.7640
Epoch  80 | Train-Loss: 0.4362 | Val-Loss: 0.5150 | Val-Acc: 0.7615
Epoch  90 | Train-Loss: 0.4221 | Val-Loss: 0.4975 | Val-Acc: 0.7628
Epoch  99 | Train-Loss: 0.3165 | Val-Loss: 0.4951 | Val-Acc: 0.7717


In [10]:
# ============================================================
# 9. Evaluation auf Testdaten
# ============================================================
net.eval()
with torch.no_grad():
    outputs = net(X_test_t)
    _, predicted = torch.max(outputs, 1)
    accuracy = accuracy_score(y_test_t, predicted)
    print(f'Testgenauigkeit: {accuracy:.4f}')

Testgenauigkeit: 0.7694


In [11]:
# ============================================================
# 10. Modell und Scaler abspeichern
# ============================================================
torch.save(net.state_dict(), 'wine_net.pth')
joblib.dump(le, 'label_encoder_wine.pkl')
joblib.dump(scaler, 'scaler_wine.pkl')   # WICHTIG: Scaler muss mitgespeichert werden!
print("Gespeichert: wine_net.pth, label_encoder_wine.pkl, scaler_wine.pkl")

Gespeichert: wine_net.pth, label_encoder_wine.pkl, scaler_wine.pkl


In [12]:
# ============================================================
# 11. Wiederladen und Vorhersage (Inferenz)
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib
import numpy as np

# Modell, Scaler und LabelEncoder laden
le     = joblib.load('label_encoder_wine.pkl')
scaler = joblib.load('scaler_wine.pkl')

net = nn.Sequential(
    nn.Linear(11, 32),
    nn.ReLU(),
#    nn.Linear(32, 16),
#    nn.ReLU(),
    nn.Linear(32, len(le.classes_))
)
net.load_state_dict(torch.load('wine_net.pth', map_location='cpu'))
net.eval()

for k, v in net.named_parameters():
    print(k, v.shape)

0.weight torch.Size([32, 11])
0.bias torch.Size([32])
2.weight torch.Size([2, 32])
2.bias torch.Size([2])


In [13]:
# ============================================================
# 12. Einzelvorhersage mit Benutzereingabe
# ============================================================
feature_names = [
    'Fixed Acidity', 'Volatile Acidity', 'Citric Acid',
    'Residual Sugar', 'Chlorides', 'Free Sulfur Dioxide',
    'Total Sulfur Dioxide', 'Density', 'pH', 'Sulphates', 'Alcohol'
]

print("Geben Sie die 11 Merkmale ein:")
values = []
for name in feature_names:
    val = float(input(f"{name}: "))
    values.append(val)

# WICHTIG: Eingabe muss denselben Scaler durchlaufen!
inputs_scaled = scaler.transform(np.array([values]))
inputs_tensor = torch.tensor(inputs_scaled, dtype=torch.float32)

with torch.no_grad():
    output = net(inputs_tensor)
    probabilities = F.softmax(output, dim=1)
    _, predicted = torch.max(output, 1)

print(f"\nVorhergesagte Qualitätsstufe: {le.inverse_transform([predicted.item()])[0]}")
print(f"Klassenwahrscheinlichkeiten:")
for cls, prob in zip(le.classes_, probabilities.numpy()[0]):
    print(f"  Qualität {cls}: {prob:.4f}")

Geben Sie die 11 Merkmale ein:

Vorhergesagte Qualitätsstufe: 0
Klassenwahrscheinlichkeiten:
  Qualität 0: 1.0000
  Qualität 1: 0.0000
